In [3]:
import pandas as pd
import os
import configparser
from neo4j import GraphDatabase
import textwrap

In [4]:
# Sử dụng file ini cho thông tin đăng nhập, nếu không thì cung cấp mặc định
HOST = 'neo4j://localhost'
USERNAME = 'neo4j'
DATABASE = 'neo4j'
PASSWORD = 'password'

NEO4J_CONF_FILE = 'neo4j.ini'

if NEO4J_CONF_FILE is not None and os.path.exists(NEO4J_CONF_FILE):
    config = configparser.RawConfigParser()
    config.read(NEO4J_CONF_FILE)
    HOST = config['NEO4J']['HOST']
    DATABASE = config['NEO4J'].get('DATABASE', 'neo4j')
    USERNAME = config['NEO4J'].get('USERNAME', DATABASE)
    PASSWORD = config['NEO4J']['PASSWORD']
    print('Using custom database properties')
else:
    print('Could not find database properties file, using defaults')

Using custom database properties


In [5]:
# Tạo driver kết nối
driver = GraphDatabase.driver(HOST, auth=(USERNAME, PASSWORD))

In [6]:
# Hàm bổ trợ
def run(driver, query, params=None):
    with driver.session(database=DATABASE) as session:
        if params is not None:
            return [r for r in session.run(query, params)]
        else:
            return [r for r in session.run(query)]

# Các truy vấn

## Tìm thông tin liên quan của một POI

In [7]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi {name: "Vạn Hạnh Mall"})
    MATCH (poi)-[:BELONGS_TO]->(category:Category)
    MATCH (poi)-[:LOCATED_AT]->(region:Region)
    RETURN  poi.id AS ID_POI, 
            poi.name AS TenPOI, 
            category.name AS DanhMuc,
            region.name AS Vung,
            poi.description AS PoiDescription, 
            poi.address AS DiaChiPOI, 
            poi.avgRating AS PoiRating, 
            poi.duration AS ThoiLuongPOI, 
            poi.openingHours AS PoiOpeningHours, 
            poi.price AS GiaPOI,
            poi.numReviews AS PoiNumReviews,
            poi.numReviews_1 AS PoiNumRating1,
            poi.numReviews_2 AS PoiNumRating2,
            poi.numReviews_3 AS PoiNumRating3,
            poi.numReviews_4 AS PoiNumRating4,
            poi.numReviews_5 AS PoiNumRating5,
            poi.url AS TripAdvisorUrl
    """)
)

[<Record ID_POI=17722137 TenPOI='Vạn Hạnh Mall' DanhMuc='Trung tâm mua sắm' Vung='Thành phố Hồ Chí Minh' PoiDescription='Vạn Hạnh Mall sẽ là sự lựa chọn hoàn hảo khi kết hợp giữa trang trí lộng lẫy trong những ngày nghỉ lễ, các sự kiện & khuyến mãi hấp dẫn đáp ứng nhu cầu của mỗi người cùng những trải nghiệm tuyệt vời từ các dịch vụ vượt trội của mình. Là điểm đến duy nhất tại Quận 10, mang đến những trải nghiệm mua sắm, ăn uống và giải trí độc đáo nhất' DiaChiPOI='11 Sư Vạn Hạnh, Phường Hòa Hưng, Thành phố Hồ Chí Minh' PoiRating=4.3 ThoiLuongPOI='1-2 giờ' PoiOpeningHours='Mo-Su 09:30-22:00' GiaPOI=0.0 PoiNumReviews=18 PoiNumRating1=1 PoiNumRating2=1 PoiNumRating3=0 PoiNumRating4=6 PoiNumRating5=10 TripAdvisorUrl='https://www.tripadvisor.com.vn/Attraction_Review-g293925-d17722137-Reviews-Van_Hanh_Mall-Ho_Chi_Minh_City.html'>]

## Tìm tất cả thông tin của một người dùng (user)

In [8]:
run(driver, textwrap.dedent("""\
    MATCH (user:User {id: 48})-[:FROM]-(origin:Origin)
    RETURN  user.id AS ID_NguoiDung, 
            user.name AS TenNguoiDung, 
            origin.name AS QuocTich
    """)
)

[<Record ID_NguoiDung=48 TenNguoiDung='tranthinam' QuocTich='Hà Tĩnh, Việt Nam'>]

## Tìm các điểm tham quan (POI) có danh mục trải nghiệm "Chuyến tham quan bằng tàu cao tốc tại Thành phố Hồ Chí Minh"

In [9]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:BELONGS_TO]->(category:Category {name: "Nhà thờ & nhà thờ lớn"})
    RETURN poi.name AS TenPOI, category.name AS DanhMucName
    """)
)

[<Record TenPOI='Nhà thờ Đức Bà Sài Gòn' DanhMucName='Nhà thờ & nhà thờ lớn'>,
 <Record TenPOI='Church of St. Joan of Arc' DanhMucName='Nhà thờ & nhà thờ lớn'>,
 <Record TenPOI='Dong Quang Parish Church' DanhMucName='Nhà thờ & nhà thờ lớn'>,
 <Record TenPOI='Carmelite Monastery of Saigon' DanhMucName='Nhà thờ & nhà thờ lớn'>,
 <Record TenPOI='The Church of Jesus Christ of Latter-day Saints' DanhMucName='Nhà thờ & nhà thờ lớn'>,
 <Record TenPOI='Hội Thánh Tin Lành Gia Định' DanhMucName='Nhà thờ & nhà thờ lớn'>]

## Liệt kê tất cả các điểm tham quan ở Vùng "Outram" cùng với địa chỉ của chúng

In [10]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:LOCATED_AT]->(region:Region {name: "Quận Tân Phú"})
    RETURN poi.name AS TenPOI, region.name AS VungName, poi.address AS DiaChiPOI
    """)
)

[<Record TenPOI='Aeon Mall Tan Phu Celadon Shopping Center' VungName='Quận Tân Phú' DiaChiPOI='30 Lối ra, Phường Tân Sơn Nhì, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Menas Mall Saigon Airport' VungName='Quận Tân Phú' DiaChiPOI='925A Trường Chinh, Phường Tây Thạnh, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Công viên Grand Park' VungName='Quận Tân Phú' DiaChiPOI='925A Trường Chinh, Phường Tây Thạnh, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Pandora City Shopping Mall' VungName='Quận Tân Phú' DiaChiPOI='1/1 Trường Chinh, Phường Tây Thạnh, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Reunification Palace. Hcmc. Vietnam' VungName='Quận Tân Phú' DiaChiPOI='925A Trường Chinh, Phường Tây Thạnh, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Công Viên Văn Hóa Lê Thị Riêng' VungName='Quận Tân Phú' DiaChiPOI='Yên Đổ, Phường Phú Thọ Hòa, Quận Tân Phú, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Vincom Mega Mall Grand Park' VungName='Quậ

## Liệt kê khoảng thời gian tham chiếu của một điểm tham quan cụ thể ("Dinh Độc Lập")

In [11]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi {name: "Dinh Độc Lập"})
    RETURN poi.name AS TenPOI, poi.duration AS KhoangThoiGian
    """)
)

[<Record TenPOI='Dinh Độc Lập' KhoangThoiGian='Dưới 1 giờ'>]

## Liệt kê các "Phòng trưng bày nghệ thuật tại Thành phố Hồ Chí Minh"

In [12]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:BELONGS_TO]->(category:Category {name: "Phòng trưng bày nghệ thuật"})
    RETURN poi.name AS TenPOI, category.name AS DanhMucName
    """)
)

[<Record TenPOI='Réhahn Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Green Palm Gallery - District 1, HCMC' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Vietnam Silver House' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Phòng Tranh Minh Anh' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Vietnam ART Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Craig Thomas Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Galerie Quynh' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Bến Thành Art & Frame' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Lotus Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Vy Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Boarc Gallery' DanhMucName='Phòng trưng bày nghệ thuật'>,
 <Record TenPOI='Tây Sơn' DanhMucName='Phòng trưng bày nghệ thuật'>,
 

## Liệt kê các "Chuyến tham quan"

In [13]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:BELONGS_TO]->(category:Category)
    WHERE category.name IN [
        "Chuyến tham quan",
        "Chuyến tham quan bằng xe buýt tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan bằng xe điện hai bánh (segway) tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan ngắm cảnh tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan nhà máy tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan & hoạt động tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan & nếm rượu vang  tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan bằng thuyền & thể thao dưới nước tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan bằng tàu cao tốc tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan bằng tàu ngầm tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan cà phê & trà tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan cưỡi ngựa tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan mua sắm tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan ngắm vịt tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan nhà máy rượu tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan quán bar",
        "Chuyến tham quan đi dạo tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan ẩm thực tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan bằng xe 4WD",
        "Chuyến tham quan bằng xe ngựa kéo tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan chạy bộ tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan cảm giác mạnh & mạo hiểm tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan leo núi tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan trên không tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan vượt hẻm núi & tuột núi tại Thành phố Hồ Chí Minh",
        "Chuyến tham quan đi bộ & cắm trại tại Thành phố Hồ Chí Minh"
        ]
    RETURN DISTINCT poi.name AS TenPOI, category.name AS DanhMucName
    """)
)

[<Record TenPOI='CHAO SHOW – Dau An Show Co., Ltd' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Backstreet Tours Saigon' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Boosh Rooftop Bar - Drinks And Games' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Emma OnTheGo' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Saigon Homies Tours' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Elixir Lounge Bar' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Vietnam Green Travel' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='The Mystic Night Show' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Specialist guide in Ho chi minh city' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Saigon Social' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Indochina Travel Group' DanhMucName='Chuyến tham quan quán bar'>,
 <Record TenPOI='Asmr Cocktails Bar' DanhMucName='Chuyến tham quan quán bar'>,

## Liệt kê các điểm tham quan có điểm đánh giá trung bình là 4

In [14]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi {avgRating:4})
    RETURN poi.name AS TenPOI, poi.avgRating AS DiemTrungBinhPOI
    """)
)

[<Record TenPOI='Bưu điện trung tâm Sài Gòn' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Dinh Độc Lập' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Saigon Centre' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Landmark 81 SkyView' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='President ho Chi Minh Statue' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Mariamman Hindu Temple' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Saigon Central Mosque' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Sân Golf Tân Sơn Nhất - Tp. Hồ Chí Minh, Việt Nam' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Giac Lam Pagoda' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Nhà thờ Huyện Sĩ' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Skybar thư giãn' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Saigon Riverfront Park' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Foot Massage Salon Quỳnh Như 137' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Quán cafe EON' DiemTrungBinhPOI=4.0>,
 <Record TenPOI='Vien Dong Massage' DiemTrungBinhPOI=4.0>,
 <Record TenPOI="L'Usine Shop" DiemTrungBinhPOI=4

## Liệt kê tất cả các điểm tham quan cùng nằm lân cận với một điểm tham quan cụ thể ("Dinh Độc Lập")

In [17]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi {name: "Dinh Độc Lập"})-[r:NEARBY]->(nearby:Poi)
    RETURN poi.name AS TenPOI, nearby.name AS TenPOILanCan, r.distance_km AS KhoangCach_Km
    ORDER BY r.distance_km ASC
    """)
)

[<Record TenPOI='Dinh Độc Lập' TenPOILanCan='Vietnam Guide Pass' KhoangCach_Km=0.02>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Dinh Design Store - Independence Palace Souvenir Shop' KhoangCach_Km=0.04>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Saigon Hotpot - Travel With Local Youth' KhoangCach_Km=0.15>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Cne Tailoring' KhoangCach_Km=0.18>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Vietnam Tourist Sim' KhoangCach_Km=0.24>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Saigon Fantaisie' KhoangCach_Km=0.25>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Ben Nghe Street Food' KhoangCach_Km=0.27>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='ACC Clinic' KhoangCach_Km=0.27>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Jessica Spa' KhoangCach_Km=0.28>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan="Nhà may Tuyển Tailor's" KhoangCach_Km=0.29>,
 <Record TenPOI='Dinh Độc Lập' TenPOILanCan='Nhà Hát Múa Rối Nước Rồng Vàng' KhoangCach_Km=0.31>,
 

## Liệt kê tất cả các điểm tham quan cùng nằm trong một vùng với một điểm tham quan cụ thể ("Dinh Độc Lập")

In [18]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi {name: "Dinh Độc Lập"})-[:LOCATED_AT]->(region:Region)
    WITH region
    MATCH (poi:Poi)-[:LOCATED_AT]->(region:Region)
    RETURN poi.name AS TenPOI, region.name AS VungName
    """)
)

[<Record TenPOI='Chợ Bến Thành' VungName='Quận 1'>,
 <Record TenPOI='Club V E-Gaming' VungName='Quận 1'>,
 <Record TenPOI='Dinh Độc Lập' VungName='Quận 1'>,
 <Record TenPOI='Đường Bùi Viện' VungName='Quận 1'>,
 <Record TenPOI='Ben Nghe Street Food' VungName='Quận 1'>,
 <Record TenPOI='Nhà Hát Múa Rối Nước Rồng Vàng' VungName='Quận 1'>,
 <Record TenPOI='Ho Chi Minh City Museum of Fine Arts' VungName='Quận 1'>,
 <Record TenPOI='Chùa Ngọc Hoàng (Phước Hải Tự)' VungName='Quận 1'>,
 <Record TenPOI='Phạm Ngũ Lão' VungName='Quận 1'>,
 <Record TenPOI='Taka Plaza' VungName='Quận 1'>,
 <Record TenPOI='Central Market' VungName='Quận 1'>,
 <Record TenPOI='Chợ Đêm Bến Thành' VungName='Quận 1'>,
 <Record TenPOI='Mariamman Hindu Temple' VungName='Quận 1'>,
 <Record TenPOI='Stressmama' VungName='Quận 1'>,
 <Record TenPOI='Hạ Spa - Massage Hochiminh City' VungName='Quận 1'>,
 <Record TenPOI='Dan Sinh Market' VungName='Quận 1'>,
 <Record TenPOI='Antique Street' VungName='Quận 1'>,
 <Record TenPOI='Ch

## Tìm những người dùng đã viết đánh giá với điểm số là 5

In [19]:
run(driver, textwrap.dedent("""\
    MATCH (user:User)-[:WROTE]->(review:Review {rating: 5})-[:RATED ]->(poi:Poi)
    RETURN user.name AS NguoiDung, review.title AS TieuDeDanhGia, poi.name AS TenPOI
    """)
)

[<Record NguoiDung='OnAir50792827286' TieuDeDanhGia='Lần đầu thăm quan bảo tàng Chứng tích chiến tranh' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Paradise07663862175' TieuDeDanhGia='goodjob' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='l_ng_c_2026' TieuDeDanhGia='Chứng tích chiến tranh chiến đi bảo tàng' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Trip49632276562' TieuDeDanhGia='Tốt' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Adventure19385216773' TieuDeDanhGia='Trải nghiệm' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Go51807767602' TieuDeDanhGia='rất tốt ạ' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Voyager16834915160' TieuDeDanhGia='Yêu Việt Nam' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='Wander12950578217' TieuDeDanhGia='Nên đến 1 lần trong đời' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='DayTrip20144423861' TieuDeDanhGia='Bảo tàng Chứng tích Chiến tranh' TenPOI='War Remnants Museum'>,
 <Record NguoiDung='h_uduyt2026' TieuD

## Tìm các POI được đánh giá cao nhất

In [20]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)<-[r:RATED]-(review:Review)
    WITH poi, AVG(review.rating) AS avgRating
    ORDER BY avgRating DESC
    RETURN poi.name AS TenPOI, avgRating AS DiemTrungBinh
    LIMIT 10
    """)
)

[<Record TenPOI='Vincom Mega Mall Thảo Điền' DiemTrungBinh=5.0>,
 <Record TenPOI='Lion Club' DiemTrungBinh=5.0>,
 <Record TenPOI='Stressmama' DiemTrungBinh=5.0>,
 <Record TenPOI='Hum Spa' DiemTrungBinh=5.0>,
 <Record TenPOI='Cosmo Club' DiemTrungBinh=5.0>,
 <Record TenPOI='Green Palm Gallery - District 1, HCMC' DiemTrungBinh=5.0>,
 <Record TenPOI='Saigon Skydeck' DiemTrungBinh=5.0>,
 <Record TenPOI='La Victoire E-Gaming Club' DiemTrungBinh=5.0>,
 <Record TenPOI='Réhahn Gallery' DiemTrungBinh=5.0>,
 <Record TenPOI="Phương's Retro Bar" DiemTrungBinh=5.0>]

## Tìm những người dùng đã đánh giá một POI cụ thể

In [21]:
run(driver, textwrap.dedent("""\
    MATCH (user:User)-[:WROTE]->(review:Review)-[:RATED]->(poi:Poi {name: 'Dinh Độc Lập'})
    RETURN user.name AS NguoiDung, review.title AS TieuDeDanhGia, review.rating AS DiemDanhGia
    """)
)

[<Record NguoiDung='toiyeuThanhHoa36' TieuDeDanhGia='Địa điểm tuyệt vời đáng ghé' DiemDanhGia=5.0>,
 <Record NguoiDung='Paradise28319823813' TieuDeDanhGia='Dinh Độc Lập, chứng nhân lịch sử' DiemDanhGia=5.0>,
 <Record NguoiDung='Sherpa21830167709' TieuDeDanhGia='Xớn xác' DiemDanhGia=1.0>,
 <Record NguoiDung='591tuen' TieuDeDanhGia='Bình thường' DiemDanhGia=3.0>,
 <Record NguoiDung='thangpham12' TieuDeDanhGia='Kiến trúc độc đáo' DiemDanhGia=5.0>,
 <Record NguoiDung='120ph_mt' TieuDeDanhGia='Rất ấn tượng' DiemDanhGia=5.0>,
 <Record NguoiDung='602lucn' TieuDeDanhGia='KHÔI NGUYÊN' DiemDanhGia=5.0>,
 <Record NguoiDung='Stay23862204542' TieuDeDanhGia='Kỉ niệm của tôi' DiemDanhGia=5.0>,
 <Record NguoiDung='lud868' TieuDeDanhGia='Nơi chứng kiến lịch sử biển hùng của dân tộc' DiemDanhGia=5.0>,
 <Record NguoiDung='ngyenanh2021' TieuDeDanhGia='cảm nhận cá nhân' DiemDanhGia=5.0>,
 <Record NguoiDung='caztushouse' TieuDeDanhGia='Cảm nhận ngắn về Dinh Độc Lập' DiemDanhGia=5.0>,
 <Record NguoiDung='ngv

## Tìm Top 10 người dùng hoạt động tích cực nhất với nhiều đánh giá nhất

In [22]:
run(driver, textwrap.dedent("""\
    MATCH (user:User)-[:WROTE]->(review:Review)
    WITH user, COUNT(*) AS reviewCount
    ORDER BY reviewCount DESC
    RETURN user.name AS TenNguoiDung, user.id AS ID_NguoiDung, reviewCount AS SoLuongDanhGia
    """)
)

[<Record TenNguoiDung='TRANDUCDANTHINH' ID_NguoiDung=181 SoLuongDanhGia=80>,
 <Record TenNguoiDung='thanhthiftu2' ID_NguoiDung=473 SoLuongDanhGia=24>,
 <Record TenNguoiDung='hai_dang_ng' ID_NguoiDung=481 SoLuongDanhGia=24>,
 <Record TenNguoiDung='DavidDean78' ID_NguoiDung=1166 SoLuongDanhGia=21>,
 <Record TenNguoiDung='BaKhoiLe' ID_NguoiDung=51 SoLuongDanhGia=18>,
 <Record TenNguoiDung='158vinht' ID_NguoiDung=176 SoLuongDanhGia=18>,
 <Record TenNguoiDung='Travelling_SE_Asia' ID_NguoiDung=1244 SoLuongDanhGia=17>,
 <Record TenNguoiDung='Jacky_Bhagat' ID_NguoiDung=1056 SoLuongDanhGia=16>,
 <Record TenNguoiDung='HoaBinhTT' ID_NguoiDung=52 SoLuongDanhGia=16>,
 <Record TenNguoiDung='highimpactuk' ID_NguoiDung=576 SoLuongDanhGia=15>,
 <Record TenNguoiDung='kmratesplates' ID_NguoiDung=673 SoLuongDanhGia=15>,
 <Record TenNguoiDung='KingMins' ID_NguoiDung=484 SoLuongDanhGia=14>,
 <Record TenNguoiDung='SshuddhoGhosh' ID_NguoiDung=711 SoLuongDanhGia=14>,
 <Record TenNguoiDung='kimmytran89' ID_Nguo

## Tìm số lượng đánh giá cho mỗi mức điểm đánh giá

In [23]:
run(driver, textwrap.dedent("""\
    MATCH (review:Review)
    RETURN review.rating AS DiemDanhGia, COUNT(*) AS SoLuongDanhGia
    ORDER BY DiemDanhGia DESC
    """)
)

[<Record DiemDanhGia=5.0 SoLuongDanhGia=19069>,
 <Record DiemDanhGia=4.0 SoLuongDanhGia=2571>,
 <Record DiemDanhGia=3.0 SoLuongDanhGia=925>,
 <Record DiemDanhGia=2.0 SoLuongDanhGia=328>,
 <Record DiemDanhGia=1.0 SoLuongDanhGia=1215>]

## Tìm những người dùng đã đánh giá các POI trong một vùng cụ thể

In [24]:
run(driver, textwrap.dedent("""\
    MATCH (user:User)-[:WROTE]->(review:Review)-[:RATED]->(poi:Poi)-[:LOCATED_AT]->(region:Region {name: 'Quận 1'})
    RETURN user.name AS NguoiDung, poi.name AS TenPOI, region.name AS Vung
    """)
)

[<Record NguoiDung='startourvietnam' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='591tuen' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='Safari20928859586' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='Annguyenthuy' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='Journey60556389670' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='nh_ph_ngp2024' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='Passenger13696406999' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='ThanhTraveler11' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='lud868' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='GoPlaces62229730183' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='tut413' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='h_ob798' TenPOI='Chợ Bến Thành' Vung='Quận 1'>,
 <Record NguoiDung='Wander30601987708' TenPOI='Chợ Bến Thành' Vung='Quận 1

## Tìm các POI có số lượng đánh giá nhiều nhất

In [25]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)<-[rated:RATED]-(:Review)
    WITH poi, COUNT(rated) AS numReviews
    ORDER BY numReviews DESC
    RETURN poi.name AS TenPOI, numReviews AS SoLuongDanhGia
    """)
)

[<Record TenPOI='Vietnam Adventure Tours' SoLuongDanhGia=1110>,
 <Record TenPOI='Deluxe Group Tours' SoLuongDanhGia=1015>,
 <Record TenPOI='CHAO SHOW – Dau An Show Co., Ltd' SoLuongDanhGia=433>,
 <Record TenPOI='Vespa Adventures' SoLuongDanhGia=387>,
 <Record TenPOI='Vietnam Travel Group' SoLuongDanhGia=380>,
 <Record TenPOI='XO Tours' SoLuongDanhGia=360>,
 <Record TenPOI='Công ty cổ phần xe khách Phương Trang' SoLuongDanhGia=318>,
 <Record TenPOI="Phan's Custom Tailor" SoLuongDanhGia=310>,
 <Record TenPOI='Euphorea Salon And Spa - Bason Branch' SoLuongDanhGia=302>,
 <Record TenPOI='TNK Travel' SoLuongDanhGia=294>,
 <Record TenPOI='Asiana Link Travel' SoLuongDanhGia=294>,
 <Record TenPOI='Tháp Tài Chính Bitexco' SoLuongDanhGia=287>,
 <Record TenPOI='Golden Lotus Healing World' SoLuongDanhGia=275>,
 <Record TenPOI='Temple Leaf Spa' SoLuongDanhGia=270>,
 <Record TenPOI='Euphorea Salon and Spa' SoLuongDanhGia=249>,
 <Record TenPOI='Ben Nghe Street Food' SoLuongDanhGia=246>,
 <Record Te

## Tìm những người dùng đã ghé thăm các POI thuộc danh mục cụ thể "Xưởng vẽ & làm đồ gốm"

In [26]:
run(driver, textwrap.dedent("""\
    MATCH (user:User)-[:WROTE]->(review:Review)-[:RATED]->(poi:Poi)-[:BELONGS_TO]->(category:Category {name: 'Xưởng vẽ & làm đồ gốm'})
    RETURN user.name AS TenNguoiDung, poi.name AS TenPOI, category.name AS DanhMucName
    """)
)

[<Record TenNguoiDung='Trail04813907654' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='628duongn' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='th_canhh2025' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='Sherpa06880572302' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='NorthStar51054685796' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='trangtO1891GD' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='Wander64464488519' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='802hangn' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='artistique49' TenPOI='NP Brows & Lashes' DanhMucName='Xưởng vẽ & làm đồ gốm'>,
 <Record TenNguoiDung='LanAnhL7' TenPOI='Spin and Gogh Art and Pot

## Tìm điểm đánh giá trung bình cho các POI ở mỗi vùng

In [27]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:LOCATED_AT]->(region:Region)
    WITH region, AVG(poi.avgRating) AS avgRating
    RETURN region.name AS Vung, avgRating AS DiemTrungBinh
    """)
)

[<Record Vung='Thành phố Hồ Chí Minh' DiemTrungBinh=2.950718889439758>,
 <Record Vung='Quận 1' DiemTrungBinh=3.1804063860667635>,
 <Record Vung='Thành phố Thủ Đức' DiemTrungBinh=3.2111111111111135>,
 <Record Vung='Quận Tân Phú' DiemTrungBinh=2.5159420289855063>,
 <Record Vung='Huyện Củ Chi' DiemTrungBinh=4.300000000000001>,
 <Record Vung='Quận 3' DiemTrungBinh=3.5451612903225804>,
 <Record Vung='Quận Bình Tân' DiemTrungBinh=2.8150000000000004>,
 <Record Vung='Quận 12' DiemTrungBinh=2.5238095238095237>,
 <Record Vung='Huyện Hóc Môn' DiemTrungBinh=1.1666666666666665>,
 <Record Vung='Quận 7' DiemTrungBinh=2.791891891891893>,
 <Record Vung='Huyện Bình Chánh' DiemTrungBinh=2.327272727272728>,
 <Record Vung='Quận 10' DiemTrungBinh=3.55>,
 <Record Vung='Quận 6' DiemTrungBinh=4.9>,
 <Record Vung='Quận 4' DiemTrungBinh=3.684615384615385>,
 <Record Vung='Quận 5' DiemTrungBinh=2.8600000000000003>,
 <Record Vung='Quận 8' DiemTrungBinh=4.2>,
 <Record Vung='Huyện Cần Giờ' DiemTrungBinh=2.93333333333

## Tìm các POI có số lượng đánh giá 5 sao nhiều nhất

In [28]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)<-[rated:RATED]-(:Review {rating: 5})
    WITH poi, COUNT(rated) AS numFiveStarRatings
    ORDER BY numFiveStarRatings DESC
    RETURN poi.name AS TenPOI, numFiveStarRatings AS SoLuongDanhGia5Sao
    """)
)

[<Record TenPOI='Vietnam Adventure Tours' SoLuongDanhGia5Sao=1055>,
 <Record TenPOI='Deluxe Group Tours' SoLuongDanhGia5Sao=972>,
 <Record TenPOI='CHAO SHOW – Dau An Show Co., Ltd' SoLuongDanhGia5Sao=425>,
 <Record TenPOI='Vespa Adventures' SoLuongDanhGia5Sao=367>,
 <Record TenPOI='Vietnam Travel Group' SoLuongDanhGia5Sao=364>,
 <Record TenPOI='XO Tours' SoLuongDanhGia5Sao=349>,
 <Record TenPOI="Phan's Custom Tailor" SoLuongDanhGia5Sao=303>,
 <Record TenPOI='Euphorea Salon And Spa - Bason Branch' SoLuongDanhGia5Sao=302>,
 <Record TenPOI='Asiana Link Travel' SoLuongDanhGia5Sao=278>,
 <Record TenPOI='TNK Travel' SoLuongDanhGia5Sao=271>,
 <Record TenPOI='Temple Leaf Spa' SoLuongDanhGia5Sao=250>,
 <Record TenPOI='Euphorea Salon and Spa' SoLuongDanhGia5Sao=247>,
 <Record TenPOI='Golden Lotus Healing World' SoLuongDanhGia5Sao=231>,
 <Record TenPOI='Kim Travel' SoLuongDanhGia5Sao=196>,
 <Record TenPOI='Monkee Shisha Lounge Saigon' SoLuongDanhGia5Sao=194>,
 <Record TenPOI='Private Daily Tours'

## Sử dụng ontology mapping để ánh xạ địa chỉ cũ và địa chỉ mới

Truy vấn dưới đây lấy ra địa chỉ cũ (được lưu trực tiếp trong thuộc tính `address` của nút `Poi`) và sử dụng quan hệ `MERGED_TO` trong ontology hành chính để tự động phân tích và tạo ra địa chỉ mới tương ứng.

In [29]:
run(driver, textwrap.dedent("""\
    MATCH (poi:Poi)-[:LOCATED_IN]->(old_w:Ward)
    MATCH (old_w)-[:BELONGS_TO]->(old_d:District)
    OPTIONAL MATCH (old_w)-[:MERGED_TO]->(new_w:Ward)
    WITH poi, old_w, old_d, coalesce(new_w, old_w) AS active_w
    MATCH (active_w)-[:BELONGS_TO]->(active_d:District)
    WITH poi, old_w, old_d, active_w, active_d,
         replace(poi.address, ", " + old_w.name + ", " + old_d.name + ", Thành phố Hồ Chí Minh", "") AS street_part
    RETURN poi.name AS TenPOI,
           poi.address AS DiaChiCu,
           street_part + ", " + active_w.name + ", " + 
           case when active_d.name = 'Thành phố Thủ Đức' then 'Thành phố Thủ Đức, Thành phố Hồ Chí Minh' else 'Thành phố Hồ Chí Minh' end AS DiaChiMoi
    LIMIT 10
    """)
)

[<Record TenPOI='Vam Sat Salt-Marsh Forest Ecological Tourist Center' DiaChiCu='Lý Nhơn, Xã An Thới Đông, Huyện Cần Giờ, Thành phố Hồ Chí Minh' DiaChiMoi='Lý Nhơn, Xã An Thới Đông, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Go Station Space' DiaChiCu='Đường tỉnh 749A, Xã Long Hòa, Huyện Cần Giờ, Thành phố Hồ Chí Minh' DiaChiMoi='Đường tỉnh 749A, Xã Cần Giờ, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='MassageGo' DiaChiCu='Đường tỉnh 749A, Xã Long Hòa, Huyện Cần Giờ, Thành phố Hồ Chí Minh' DiaChiMoi='Đường tỉnh 749A, Xã Cần Giờ, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='VietNam Unique Travelsss' DiaChiCu='345 Võ Văn Tần, Phường 5, Quận 3, Thành phố Hồ Chí Minh' DiaChiMoi='345 Võ Văn Tần, Phường Bàn Cờ, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Saigonbiketours' DiaChiCu='352/6 Nguyễn Đình Chiểu, Phường 4, Quận 3, Thành phố Hồ Chí Minh' DiaChiMoi='352/6 Nguyễn Đình Chiểu, Phường Bàn Cờ, Thành phố Hồ Chí Minh'>,
 <Record TenPOI='Cỏ Mềm Homelab' DiaChiCu='398 Nguyễn Đình Chiểu, Phường 4, Quận 3

# Đóng kết nối driver

In [30]:
driver.close()